# 3. Mean-value expansion (Section 4.2) and Appendix B

Checks:

* Eq. (*lap*), the Laplacian of $w^s$, and Eq. (*lowmu*), symbolically (sympy);
* Eq. (*Tn*): the regrouped terms equal the terms of the mean-value expansion (Eq. *pizzetti*);
* Eq. (*contact*), the mean over the internally tangent disk;
* **Proposition 1 (Lagrange)**: the remainder lies between the bounds;
* **Proposition 2 (Abel)**;
* Eq. (*dDdz*), against a numerical derivative of the reference.

**Note on Proposition 1.** The smallest $r$ on the occulting disk is $\max(0,z-p)$, not $|z-p|$: for
$z<p$ the disk covers the stellar centre. With $|z-p|$, as in an earlier draft, the bound fails
for $z<p$ (cell 3d shows both).

In [1]:
import time
import numpy as np
import mpmath as mp
import refcheck as rc

FULL = False          # True: the larger grids (minutes to hours)
t_start = time.time()

import sympy as sp
mp.mp.dps = 40

## 3a. Symbolic checks

In [2]:
x, y, s, zz, pp = sp.symbols("x y s z p", real=True)
w = 1 - x**2 - y**2
lap = sp.diff(w**s, x, 2) + sp.diff(w**s, y, 2)
claim = 4 * s * (s - 1) * w**(s - 2) - 4 * s**2 * w**(s - 1)
pts = [(sp.Rational(3, 10), sp.Rational(1, 5), sp.Rational(37, 100)), (sp.Rational(1, 10), sp.Rational(1, 2), sp.Rational(17, 10))]
print("Eq.(lap):", max(abs(sp.N((lap - claim).subs({x: a, y: b, s: c}), 30)) for a, b, c in pts))

f = sp.sqrt(w)
terms, g_ = [], f
for n in range(4):
    if n:
        g_ = sp.diff(g_, x, 2) + sp.diff(g_, y, 2)
    terms.append(sp.Rational(1, 4**n) / (sp.factorial(n) * sp.factorial(n + 1)) * g_.subs({y: 0, x: zz}) * pp**(2 * n))
v = 1 / (1 - zz**2)
lowmu = sp.sqrt(1 - zz**2) * (1 - pp**2 / 8 * (v + v**2) + pp**4 / 192 * (v**2 + 6 * v**3 - 15 * v**4)
                              - pp**6 / 1024 * (v**3 + 15 * v**4 - 105 * v**5 + 105 * v**6))
print("Eq.(lowmu) minus the mean-value expansion through p^6:", sp.simplify(sum(terms) - lowmu))

Eq.(lap): 0


Eq.(lowmu) minus the mean-value expansion through p^6: 0


## 3b. Eq. (Tn) equals the mean-value term

In [3]:
worst = 0
for gam in [sp.Rational(3, 10), sp.Rational(1, 2), sp.Rational(17, 10)]:
    g_ = w**gam
    for n in range(3):
        if n:
            g_ = sp.diff(g_, x, 2) + sp.diff(g_, y, 2)
        for zv, pv in [(sp.Rational(1, 5), sp.Rational(1, 10)), (sp.Rational(3, 5), sp.Rational(3, 10))]:
            term = sp.N(sp.Rational(1, 4**n) / (sp.factorial(n) * sp.factorial(n + 1)) * g_.subs({y: 0, x: zv}) * pv**(2 * n), 40)
            tn = rc.T(n, mp.mpf(zv.p) / zv.q, mp.mpf(pv.p) / pv.q, mp.mpf(gam.p) / gam.q)
            worst = max(worst, abs(mp.mpf(str(term)) - tn))
print("max |T_n - mean-value term| =", mp.nstr(worst, 3))

max |T_n - mean-value term| = 2.3e-41


## 3c. Eq. (contact): the internally tangent disk

In [4]:
worst = max(abs(rc.D_mp(1 - mp.mpf(q), mp.mpf(q), 0.5, dps=25) - rc.D_tangent(mp.mpf(q))) for q in ["0.01", "0.3", "0.8"])
print("max |Eq.(contact) - reference| =", mp.nstr(worst, 3))

max |Eq.(contact) - reference| = 1.35e-36


## 3d. Proposition 1 (Lagrange remainder)

$D-S^{\rm P}_N$ must lie between the bounds built from $\Delta^{N+1}f$ at the smallest and the largest
$r$ on the disk. Remainders below $10^{-13}$ are not judged (double-precision reference).

In [5]:
res = {"max(0, z-p) (correct)": 0, "|z-p| (earlier draft)": 0}
judged = 0
gammas = [0.25, 0.5, 0.75] if FULL else [0.5]
for gam in gammas:
    for p in [0.01, 0.05, 0.1, 0.2]:
        for zf in sorted(set(list(np.linspace(0, 0.98 * (1 - p), 8 if FULL else 5)) + [0.5 * p, 1.5 * p])):
            z, pm, gm = mp.mpf(zf), mp.mpf(p), mp.mpf(gam)
            d = rc.D_fast(zf, p, gam)
            for N in [0, 1, 2]:
                rem = d - rc.SP(N, z, pm, gm)
                if abs(rem) < 1e-13:
                    continue
                judged += 1
                for label, rmin in [("max(0, z-p) (correct)", None), ("|z-p| (earlier draft)", abs(z - pm))]:
                    lo, hi = rc.lagrange_bounds(N, z, pm, gm, rmin=rmin)
                    tol = 1e-15 * abs(d)
                    res[label] += not (lo - tol <= rem <= hi + tol)
print("judged:", judged)
for k, v in res.items():
    print(f"  smallest radius {k}: {v} violations")
assert res["max(0, z-p) (correct)"] == 0

judged: 80
  smallest radius max(0, z-p) (correct): 0 violations
  smallest radius |z-p| (earlier draft): 11 violations


## 3e. Proposition 2 (Abel bound), $\gamma=\tfrac12$

In [6]:
judged = viol = 0
ratios = []
for p in ([0.01, 0.05, 0.1, 0.2, 0.3] if FULL else [0.05, 0.2]):
    for zf in np.linspace(0, 0.999 * (1 - p), 10 if FULL else 5):
        z, pm, g = mp.mpf(zf), mp.mpf(p), mp.mpf(0.5)
        d = rc.D_fast(zf, p, 0.5)
        for N in [1, 3]:
            diff = rc.SP(N, z, pm, g) - d
            bound = (pm / (1 - z))**(2 * N + 2) * (rc.SP(N, z, 1 - z, g) - rc.D_tangent(1 - z))
            if bound < 1e-13:
                continue
            judged += 1
            viol += diff < -1e-15 or diff > bound * (1 + 1e-12) + 1e-15
            ratios.append(float(diff / bound))
print(f"judged {judged}, violations {viol}, max error/bound {max(ratios):.3f}")
assert viol == 0

judged 20, violations 0, max error/bound 0.990


## 3f. Eq. (dDdz) against a numerical derivative of the reference

In [7]:
worst = 0.0
for g in [0.3, 0.5, 1.7]:
    for z, p in [(0.2, 0.1), (0.5, 0.2), (0.8, 0.15)]:
        h = 1e-4
        num = (-rc.D_fast(z + 2*h, p, g) + 8*rc.D_fast(z + h, p, g) - 8*rc.D_fast(z - h, p, g) + rc.D_fast(z - 2*h, p, g)) / (12 * h)
        worst = max(worst, abs(num - float(rc.dDdz(mp.mpf(z), mp.mpf(p), mp.mpf(g)))) / abs(num))
print(f"max relative difference: {worst:.1e}  (limited by the finite difference)")
assert worst < 1e-9

max relative difference: 4.0e-12  (limited by the finite difference)


In [8]:
print(f"notebook run time: {time.time() - t_start:.0f} s")

notebook run time: 31 s
